In [2]:
import numpy as np
import matplotlib.pyplot as plt
import gym
import random

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
def build_env(slippery):
  env = gym.make("FrozenLake-v1",is_slippery=slippery,render_mode = "rgb_array",map_name = "8x8")
  env.reset()
  return env

In [4]:
def epsilon_greedy_policy(Q , state , epsilon):
  p = random.random()
  if p < epsilon:
    n_actions = Q.shape[1]
    return np.random.randint(n_actions)
  else:
    return np.argmax(Q[state])

In [ ]:
def mc_control_epsilon_greedy(env , n_episods ,gamma,epsilon):
  n_states = env.observation_space.n
  n_actions = env.action_space.n
  Q = np.zeros((n_states,n_actions) , dtype = float)
  rewards = []
  length = []
  returns = {}
  for episods in range(n_episods):
    r = 0
    l = 0
    state, info= env.reset()
    trajectory = []
    while True:
      action = epsilon_greedy_policy(Q,state,epsilon)
      next_state,reward,terminated,truncated , Info= env.step(action)
      trajectory.append((state,action,reward))
      state = next_state
      r += reward
      l += 1
      if terminated or truncated:
        break
    rewards.append(r)
    length.append(l)
    G = 0
    visited = set()
    for state, action, reward in reversed(trajectory):
        G = gamma * G + reward
        if (state, action) not in visited:
            visited.add((state, action))
            if (state, action) not in returns:
              returns[(state, action)] = []
            returns[(state, action)].append(G)
            Q[state, action] = np.mean(returns[(state, action)])
  policy = np.argmax(Q, axis=1)
  return Q , policy , rewards , length






In [6]:
def mc_off_policy_importance_sampling(env , n_episodes , gamma , epsilon):
  n_states = env.observation_space.n
  n_actions = env.action_space.n
  Q = np.zeros((n_states,n_actions))
  C = np.zeros((n_states,n_actions))
  rewards = []
  length = []
  for episode in range(n_episodes):
    r = 0
    l = 0
    state , info = env.reset()
    trajectory = []
    while True:
      a = epsilon_greedy_policy(Q,state,epsilon)
      next_state,reward,terminated,truncated , Info= env.step(a)
      trajectory.append((state,a,reward))
      state = next_state
      r += reward
      l += 1
      if terminated or truncated:
        break
    rewards.append(r)
    length.append(l)
    G = 0
    W = 1
    for state,a,reward in reversed(trajectory):
      G = gamma * G + reward
      C[state,a] += W
      Q[state,a] += (W/C[state,a]) * (G - Q[state,a])
      target_action = np.argmax(Q[state])
      if a != target_action:
        break
      b = epsilon / n_actions + (1 - epsilon)
      W = W * (1.0/b)
  policy = np.argmax(Q, axis=1)
  return Q ,policy, rewards , length


In [8]:
def plot_metrics(rewards, lengths, title="Training Metrics"):
    episodes = range(1, len(rewards) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(episodes, rewards, color='blue')
    ax1.set_xlabel("Episode")
    ax1.set_ylabel("Total Reward")
    ax1.set_title("Avg Reward vs Episode")
    ax1.grid(True, linestyle='--', alpha=0.6)

    ax2.plot(episodes, lengths, color='orange')
    ax2.set_xlabel("Episode")
    ax2.set_ylabel("Episode Length")
    ax2.set_title("Avg Episode Length vs Episode")
    ax2.grid(True, linestyle='--', alpha=0.6)

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


In [9]:
def plot_policy_on_frozen_lake(env, policy, title="FrozenLake policy"):
	desc = np.asarray(env.unwrapped.desc, dtype=str)
	policy_grid = np.asarray(policy).reshape(desc.shape)
	arrows = np.array(["<", "v", ">", "^"])
	colors = {
		"S": "#9be7a1",
		"F": "#dceefb",
		"H": "#3a3a3a",
		"G": "#ffd54f",
	}

	fig, ax = plt.subplots(figsize=(8, 8))
	for r in range(desc.shape[0]):
		for c in range(desc.shape[1]):
			tile = desc[r, c]
			rect = plt.Rectangle((c, desc.shape[0] - 1 - r), 1, 1, facecolor=colors[tile], edgecolor="black", linewidth=1.5)
			ax.add_patch(rect)

			if tile == "H":
				label = "H"
			elif tile == "G":
				label = "G"
			elif tile == "S":
				label = f"S{arrows[policy_grid[r, c]]}"
			else:
				label = arrows[policy_grid[r, c]]

			ax.text(c + 0.5, desc.shape[0] - 1 - r + 0.5, label, ha="center", va="center", fontsize=16, fontweight="bold", color="black")

	ax.set_xlim(0, desc.shape[1])
	ax.set_ylim(0, desc.shape[0])
	ax.set_xticks(np.arange(desc.shape[1] + 1))
	ax.set_yticks(np.arange(desc.shape[0] + 1))
	ax.grid(True, color="black", linewidth=1.0)
	ax.set_xticklabels([])
	ax.set_yticklabels([])
	ax.set_aspect("equal")
	ax.set_title(title)
	plt.tight_layout()
	plt.show()
